# 03 ML modeling: K Means preparation
Loads `processed/stage2_clean_join/master_county_clean.csv` from notebook 02. Existing feature selection and validation cells retained. **This notebook does not yet fit K Means.**


Run this notebook in VS Code Jupyter with the repository open. The setup locates the repository from the Jupyter working directory or `REPO_DIR_OVERRIDE`.


In [ ]:
from pathlib import Path
import subprocess

# Notebooks live in scripts/; the repository contains raw data/, run_folder/, and processed/.
REPO_DIR_OVERRIDE = None  # Optional: full local repository path if Jupyter runs elsewhere

def find_repo():
    if REPO_DIR_OVERRIDE is not None:
        root = Path(REPO_DIR_OVERRIDE).expanduser().resolve()
    else:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents]
        root = next((p for p in candidates if (p / 'run_folder').is_dir() and (p / 'raw data').is_dir()), None)
        if root is None:
            result = subprocess.run(['git', 'rev-parse', '--show-toplevel'], cwd=cwd, capture_output=True, text=True)
            if result.returncode == 0:
                root = Path(result.stdout.strip()).resolve()
        if root is None:
            raise FileNotFoundError('Jupyter is outside the repository. Open the notebook in VS Code from the repo, or set REPO_DIR_OVERRIDE.')
    if not (root / 'run_folder').is_dir() or not (root / 'raw data').is_dir():
        raise FileNotFoundError(f'Expected raw data/ and run_folder/ under {root}')
    return root

project_dir = find_repo()
RAW_DIR = project_dir / 'raw data'
RUN_DIR = project_dir / 'run_folder'
OUTPUT_DIR = project_dir / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_input(filename=None, folder=None, search_dir=None):
    base = search_dir or RUN_DIR
    if (filename is None) == (folder is None):
        raise ValueError('Specify exactly one of filename or folder')
    matches = sorted(p for p in base.rglob('*') if (p.is_file() and p.name.lower() == filename.lower()) if filename is not None) if filename else sorted(p for p in base.rglob('*') if p.is_dir() and p.name.lower() == folder.lower())
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename or folder} under {base}; found {len(matches)}: {matches[:10]}')
    return matches[0]

print('Repository:', project_dir)
print('Raw rasters:', RAW_DIR / 'CERF')
print('Cleaning inputs:', RUN_DIR)
print('Processed outputs:', OUTPUT_DIR)


In [ ]:
import pandas as pd
import numpy as np
master_path = OUTPUT_DIR / 'stage2_clean_join' / 'master_county_clean.csv'
if not master_path.exists():
    raise FileNotFoundError(f'Run 02_Dataset_Cleaning.ipynb first: {master_path}')
master = pd.read_csv(master_path, dtype={'county_fips':'string','state_fips':'string'})
print('Master rows:', len(master))

In [ ]:
# Stage 3: K Means Clustering
# Prepare the county level modeling data

import pandas as pd
import numpy as np

# Start with the cleaned master dataset
model_df = master.copy()

# Columns that should not be used as model features
exclude_cols = [
    'county_fips',
    'state_fips',
    'latitude',
    'longitude',
    'cerf_join',
    'ncsl_join',
    'atlas_join',
    'atlas_record_count',
    'has_existing_data_center'
]

# Select numerical features
numeric_cols = model_df.select_dtypes(include=np.number).columns

feature_cols = [
    col for col in numeric_cols
    if col not in exclude_cols
]

X = model_df[feature_cols].copy()

# Replace infinite values with missing values
X = X.replace([np.inf, -np.inf], np.nan)

print(f'Counties: {len(X):,}')
print(f'Features available: {len(feature_cols)}')

print('\nSelected features:')
for col in feature_cols:
    print(col)

print('\nMissing values:')
display(X.isna().sum().sort_values(ascending=False))

display(X.head())

In [ ]:
# K Means Model 1: CERF baseline features

cerf_features = [
    'commercial_electricity_rates_dollar_per_kwh_mean',
    'industrial_electricity_rates_dollar_per_kwh_mean',
    'composite_siting_suitability_mean',
    'distance_to_substation_mean',
    'distance_to_substation_min',
    'land_value_dollar_per_sqft_mean',
    'mechanical_cooling_fraction_mean',
    'personal_property_tax_rate_mean',
    'real_property_tax_rate_mean',
    'sales_tax_rate_mean',
    'water_cooling_fraction_mean'
]

# Keep only counties with complete CERF coverage
model_data = master.loc[
    master['cerf_join'] == 'both'
].copy()

X = model_data[cerf_features].copy()

# Replace infinite values with missing values
X = X.replace([np.inf, -np.inf], np.nan)

print(f'Counties available: {len(X):,}')
print(f'Features selected: {len(cerf_features)}')

print('\nMissing values:')
display(X.isna().sum())

print('\nFeature summary:')
display(X.describe().T)

In [ ]:
# Validate K Means modeling dataset

print("K Means Data Validation")
print("=" * 40)

# 1. Check dataset dimensions
print(f"Counties: {len(X):,}")
print(f"Features: {X.shape[1]}")

# 2. Check missing values
print(f"Missing values: {X.isna().sum().sum():,}")

# 3. Check infinite values
print(f"Infinite values: {np.isinf(X.to_numpy()).sum():,}")

# 4. Check duplicate counties
print(f"Duplicate counties: {model_data['county_fips'].duplicated().sum():,}")

# 5. Check feature types
print("\nFeature data types:")
print(X.dtypes.value_counts())

# 6. Check features with no variation
constant_cols = X.columns[X.nunique(dropna=True) <= 1].tolist()
print(f"\nConstant features: {constant_cols}")

# 7. Check county coverage
print("\nCERF join coverage:")
print(model_data['cerf_join'].value_counts())

# 8. Preview
display(X.head())

In [ ]:
# Final validation before scaling

assert len(X) > 0, "No counties available"
assert X.shape[1] == 11, "Unexpected number of features"
assert X.columns.tolist() == cerf_features, "Feature list does not match"
assert X.notna().all().all(), "Missing values found"
assert np.isfinite(X.to_numpy()).all(), "Infinite values found"
assert not model_data['county_fips'].duplicated().any(), "Duplicate counties found"
assert (X.nunique() > 1).all(), "Constant feature found"
assert model_data['cerf_join'].eq('both').all(), "Unmatched CERF records found"

print("All validation checks passed!")
print("Dataset is ready for scaling.")